# Model Training - ChatKasir

- Nama: Achmad Rif'an
- Bagian: AI-1 (Model Architect)

## 1. Import Library

In [ ]:
import os
import time
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model

## 2. Memuat Konfigurasi

Kita mengambil file model_config.json untuk mengetahui parameter model seperti vocab_size dan max_length. Ini penting agar arsitektur yang kita bangun di notebook ini sama persis dengan data yang sudah disiapkan.

In [ ]:
# Memuat konfigurasi arsitektur dari file JSON
config_path = "..\\assets\\data\\model_config.json"
with open(config_path, "r") as f:
    config = json.load(f)

# Mengambil variabel penting
VOCAB_SIZE = config['vocab_size']
MAX_LENGTH = config['max_length']
NUM_TAGS = config['num_product_tags']

print(f"Konfigurasi dimuat: Vocab={VOCAB_SIZE}, Max Length={MAX_LENGTH}, Num Tags={NUM_TAGS}")

## 3. Data Loading

1. Memuat Dataset: Kita memuat file dataset_chatkasir.npz yang berisi array NumPy untuk data Training, Validation, dan Testing.

2. Membuat tf.data.Dataset: Kita mengubah array tersebut menjadi objek Dataset TensorFlow. Ini adalah cara paling efisien untuk melatih model karena mendukung fitur shuffling (mengacak data) dan batching (mengambil data sedikit demi sedikit) agar tidak membebani memori RAM.

In [ ]:
# Memuat dataset yang sudah dibagi (Train, Val, Test)
data_path = "..\\assets\\data\\dataset_chatkasir.npz"
data = np.load(data_path)

# Ekstrak data Training
X_train = data['X_train']
Y_prod_train = data['Y_prod_train']
Y_qty_train = data['Y_qty_train']
Y_price_train = data['Y_price_train']

# Ekstrak data Validation
X_val = data['X_val']
Y_prod_val = data['Y_prod_val']
Y_qty_val = data['Y_qty_val']
Y_price_val = data['Y_price_val']

print(f"Dataset dimuat: Training={len(X_train)} baris, Validation={len(X_val)} baris")

In [ ]:
# Mengonversi ke tf.data.Dataset untuk efisiensi training
BATCH_SIZE = 32 # Jumlah data yang diproses sekali epoch

def create_tf_dataset(X, y_prod, y_qty, y_price, is_training=True):
    # Gabungkan Input (X) dengan 3 Target (Y)
    ds = tf.data.Dataset.from_tensor_slices((X, (y_prod, y_qty, y_price)))
    
    if is_training:
        ds = ds.shuffle(10000) # Acak data agar model tidak menghafal urutan
    
    # Ambil data per batch dan siapkan batch berikutnya di latar belakang (prefetch)
    # AUTOTUNE = otomatis mengatur penggunaan CPU/GPU
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# Buat objek dataset untuk Training dan Validation
train_ds = create_tf_dataset(X_train, Y_prod_train, Y_qty_train, Y_price_train)
val_ds = create_tf_dataset(X_val, Y_prod_val, Y_qty_val, Y_price_val, is_training=False)

print("Objek tf.data.Dataset berhasil dibuat")

## 4. Re-build Model

Meskipun kita sudah merancang model di notebook sebelumnya, kita perlu mendefinisikan ulang strukturnya di notebook ini agar objek model tersebut tercipta kembali di memori sebelum dilatih.

Ada beberapa hal penting yang kita lakukan di sini:

1. Mendefinisikan TransformerEncoder: Kita menyertakan kembali kelas kustom ini, lengkap dengan dukungan masking agar model tidak "bingung" melihat token padding.

2. Mendefinisikan model_transformer: Fungsi ini membangun arsitektur Multi-Task kita yang terdiri dari satu tulang punggung (backbone) Transformer dan tiga cabang prediksi (Produk, Jumlah, Harga).

3. Instansiasi Model: Kita memanggil fungsi tersebut menggunakan variabel VOCAB_SIZE, MAX_LENGTH, dan NUM_TAGS yang sudah kita muat dari file konfigurasi di Tahap 1.

In [ ]:
# Custom Layer Transformer dengan dukungan Masking
class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.supports_masking = True # Pastikan layer mendukung penanda padding
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim) 
        self.ffn = tf.keras.Sequential([Dense(ff_dim, activation="relu"), Dense(embed_dim)])  
        self.layernorm1 = LayerNormalization(epsilon=1e-6) 
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)  
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False, mask=None):
        # Gunakan mask agar attention mengabaikan token padding [PAD]
        padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32) if mask is not None else None
        
        attn_output = self.att(inputs, inputs, attention_mask=padding_mask)  
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)  

        ffn_output = self.ffn(out1) 
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Fungsi arsitektur model
def model_transformer(vocab_size, max_length, num_product_tags):
    embed_dim = 64  # Dimensi representasi kata
    num_heads = 4   # Jumlah mekanisme attention
    ff_dim = 128    # Kapasitas memori internal
    
    # Input Layer
    inputs = Input(shape=(max_length,), name="input_ids")  
    
    # Shared Backbone (Transformer)
    # mask_zero=True sangat penting untuk menangani padding secara otomatis
    x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True)(inputs)  
    x = TransformerEncoder(embed_dim, num_heads, ff_dim)(x) 
    x_pooled = GlobalAveragePooling1D()(x) # Ringkasan kalimat untuk regresi
    
    # Cabang 1: Produk (NER) - Memprediksi tag untuk setiap kata
    branch_product = Dense(64, activation='relu')(x)
    output_product = Dense(num_product_tags, activation='softmax', name="product_tags")(branch_product)
    
    # Cabang 2: Jumlah (Quantity) - Regresi nilai angka jumlah pesanan
    branch_quantity = Dense(32, activation='relu')(x_pooled)
    output_quantity = Dense(1, activation='relu', name="quantity")(branch_quantity)
    
    # Cabang 3: Harga (Price) - Regresi nilai harga satuan
    branch_price = Dense(32, activation='relu')(x_pooled)
    output_price = Dense(1, activation='relu', name="price")(branch_price)
    
    return Model(inputs=inputs, outputs=[output_product, output_quantity, output_price])

# Merakit model menggunakan parameter dari konfigurasi Tahap 1
model = model_transformer(
    vocab_size=VOCAB_SIZE,
    max_length=MAX_LENGTH,
    num_product_tags=NUM_TAGS
)

# Tampilkan ringkasan arsitektur
model.summary()

## 5. Loss Function & Optimizer
1. Loss Function: Ini adalah rumus matematika untuk menghitung seberapa jauh tebakan model dari kenyataan.

   - Cabang Produk: Menggunakan Sparse Categorical Crossentropy karena tugasnya adalah klasifikasi kategori (O, B-PROD, I-PROD).

   - Cabang Quantity: Menggunakan Mean Squared Error (MSE) karena tugasnya menebak angka kontinu.

   - Cabang Harga (MaskedPriceLoss): Ini yang paling spesial. Kita harus membuat fungsi kustom agar model mengabaikan data yang nilai harganya -1 (saat harga tidak disebutkan di chat). Jika tidak di-masking, model akan belajar menebak angka -1, padahal itu hanya penanda data kosong.

2. Optimizer: Ini adalah algoritma yang bertugas memperbaiki bobot model berdasarkan nilai loss. Kita menggunakan Adam, yang merupakan standar industri karena kecepatannya dalam belajar.

3. Dynamic Weighting Variables: Kita menyiapkan variabel pembobot awal agar nantinya model bisa menyeimbangkan fokus belajarnya antara Produk, Jumlah, dan Harga secara otomatis.

In [ ]:
# Custom Loss untuk harga satuan
class MaskedPriceLoss(tf.keras.losses.Loss):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Menggunakan Mean Squared Error sebagai dasar perhitungan
        self.mse = tf.keras.losses.MeanSquaredError(reduction='none')

    def call(self, y_true, y_pred):
        # Buat masker: abaikan jika y_true bernilai -1
        mask = tf.cast(tf.not_equal(y_true, -1.0), tf.float32)
        
        # Hitung MSE mentah
        loss = self.mse(y_true, y_pred)
        
        # Kalikan loss dengan masker (loss jadi 0 untuk data bernilai -1)
        masked_loss = loss * mask
        
        # Kembalikan rata-rata loss hanya dari data yang valid
        return tf.reduce_sum(masked_loss) / (tf.reduce_sum(mask) + 1e-7)

# Inisialisasi Loss Function untuk tiap cabang
# Sparse karena label produk berbentuk angka ID (0, 1, 2)
loss_fn_product = tf.keras.losses.SparseCategoricalCrossentropy()
loss_fn_quantity = tf.keras.losses.MeanSquaredError()
loss_fn_price = MaskedPriceLoss()

# Inisialisasi Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# Variabel untuk Dynamic Loss Weighting
# bobot awal seimbang (1.0) untuk ketiga tugas
w_product = tf.Variable(1.0, trainable=False, name="w_prod")
w_quantity = tf.Variable(1.0, trainable=False, name="w_qty")
w_price = tf.Variable(1.0, trainable=False, name="w_price")

## 6. Custom Training Loop (tf.GradientTape)

Membuat 2 fungsi utama:

1. train_step: Ini adalah rutinitas belajar model. Di sini, kita menggunakan tf.GradientTape sebagai "perekam". Saat model membuat tebakan (Forward Pass), tape merekamnya. Lalu kita hitung kesalahannya (Loss), dan kita putar balik rekaman tersebut (Backpropagation) untuk memperbaiki saraf-saraf model menggunakan Optimizer. Kita juga menerapkan Dynamic Loss Weighting dengan mengalikan loss dengan bobot w_product, w_quantity, dan w_price.

2. val_step: Ini adalah rutinitas ujian/tryout. Tidak ada tape perekam dan tidak ada perbaikan saraf. Model hanya murni menebak, lalu kita hitung seberapa meleset tebakannya.

In [ ]:
# FUNGSI TRAINING
@tf.function
def train_step(x_batch, y_prod, y_qty, y_price):
    # Buka GradientTape
    with tf.GradientTape() as tape:
        
        # Forward Pass: model memprediksi
        # training=True, agar Dropout dll menyala
        pred_prod, pred_qty, pred_price = model(x_batch, training=True)
        
        # Hitung seberapa meleset prediksinya
        loss_prod = loss_fn_product(y_prod, pred_prod)
        loss_qty = loss_fn_quantity(y_qty, pred_qty)
        loss_price = loss_fn_price(y_price, pred_price)
        
        # Terapkan Dynamic Loss Weighting
        # Kalikan dengan bobot masing-masing sebelum digabungkan
        weighted_loss_prod = w_product * loss_prod
        weighted_loss_qty = w_quantity * loss_qty
        weighted_loss_price = w_price * loss_price
        
        total_loss = weighted_loss_prod + weighted_loss_qty + weighted_loss_price
        
    # Hitung gradien (arah perbaikan)
    gradients = tape.gradient(total_loss, model.trainable_weights)
    
    # Terapkan perbaikan bobot ke model menggunakan Optimizer
    optimizer.apply_gradients(zip(gradients, model.trainable_weights))
    
    # Kembalikan semua nilai loss untuk dipantau di layar
    return total_loss, loss_prod, loss_qty, loss_price

# FUNGSI VALIDATION
@tf.function
def val_step(x_batch, y_prod, y_qty, y_price):
    # Forward Pass saja
    # training=False, agar model memprediksi dengan kekuatan penuh (tanpa Dropout)
    pred_prod, pred_qty, pred_price = model(x_batch, training=False)
    
    # Hitung Loss mentah
    loss_prod = loss_fn_product(y_prod, pred_prod)
    loss_qty = loss_fn_quantity(y_qty, pred_qty)
    loss_price = loss_fn_price(y_price, pred_price)
    
    # Terapkan pembobotan untuk evaluasi
    weighted_loss_prod = w_product * loss_prod
    weighted_loss_qty = w_quantity * loss_qty
    weighted_loss_price = w_price * loss_price
    
    total_loss = weighted_loss_prod + weighted_loss_qty + weighted_loss_price
    
    return total_loss, loss_prod, loss_qty, loss_price

print("Fungsi train_step dan val_step berhasil dibuat")

## 7. Model Training

Pada tahap ini, kita akan menjalankan proses pelatihan selama beberapa Epoch. Di setiap epoch, model akan melakukan 2 hal besar:

1. Siklus Pelatihan (Training): Model membaca seluruh data di train_ds, melakukan prediksi, menghitung kesalahan, dan memperbaiki dirinya sendiri menggunakan fungsi train_step yang sudah kita buat dengan tf.GradientTape.

2. Siklus Evaluasi (Validation): Setelah satu putaran belajar selesai, model diuji menggunakan data val_ds melalui fungsi val_step untuk melihat sejauh mana ia bisa menggeneralisasi pola tanpa melakukan perbaikan bobot.

Kita juga akan memantau nilai loss dari ketiga cabang (Produk, Quantity, dan Price) untuk memastikan bahwa Dynamic Loss Weighting dan MaskedPriceLoss bekerja dengan efektif dalam menyeimbangkan prioritas belajar model.

In [ ]:
import time

# KONFIGURASI PELATIHAN
EPOCHS = 10  # Jumlah putaran pelatihan
history = [] # Untuk menyimpan catatan progres loss

print(f"Memulai Pelatihan selama {EPOCHS} Epoch...\n")

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # 1. TRAINING
    # Inisialisasi pengumpul loss untuk satu epoch
    epoch_train_loss = 0.0
    num_train_batches = 0
    
    for x_batch, (y_prod, y_qty, y_price) in train_ds:
        # Jalankan 1 langkah training
        t_loss, l_prod, l_qty, l_price = train_step(x_batch, y_prod, y_qty, y_price)
        
        epoch_train_loss += t_loss
        num_train_batches += 1
        
    avg_train_loss = epoch_train_loss / num_train_batches

    # 2. VALIDATION 
    epoch_val_loss = 0.0
    num_val_batches = 0
    
    for x_batch_val, (y_prod_val, y_qty_val, y_price_val) in val_ds:
        # Jalankan 1 langkah validation (tanpa update bobot)
        v_loss, _, _, _ = val_step(x_batch_val, y_prod_val, y_qty_val, y_price_val)
        
        epoch_val_loss += v_loss
        num_val_batches += 1
        
    avg_val_loss = epoch_val_loss / num_val_batches

    # 3. MONITORING
    duration = time.time() - start_time
    print(f"Epoch {epoch+1}/{EPOCHS} - {duration:.1f}s")
    print(f" > Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Simpan progres untuk diplot nanti di notebook 03
    history.append({'train': avg_train_loss.numpy(), 'val': avg_val_loss.numpy()})

## 8. Simpan Model
Menyimpan Model dengan format 
1. Keras yang menyimpan seluruh konfigurasi model (arsitektur), bobot, dan konfigurasi optimizer dalam 1 file tunggal
2. SavedModel yang merupakan format asli dari TensorFlow, dan menghasilkan sebuah folder yang berisi file .pb (Protocol Buffer) serta sub-folder lainnya

In [ ]:
# Buat folder models
os.makedirs("..\\assets\\models", exist_ok=True)
save_path_keras = "..\\assets\\models\\chatkasir_model.keras"

# Simpan seluruh arsitektur dan bobot model
model.save(save_path_keras)

# Tentukan folder penyimpanan
save_path_sm = "..\\assets\\models\\chatkasir_saved_model"

# Simpan model sebagai SavedModel
model.save(save_path_sm, save_format="tf")

print(f"\nPelatihan Selesai. Model disimpan di: {save_path_keras} dan {save_path_sm}")